In [14]:

# 1. Import
import numpy as np
import pandas as pd


# 2. Load
df = pd.read_csv('/kaggle/input/datasets/durjoychandrapaul/house-price-bangladesh/house_price_bd.csv')


# 3. Clean Price
df['Price_in_taka'] = df['Price_in_taka'].str.replace('৳', '')
df['Price_in_taka'] = df['Price_in_taka'].str.replace(',', '')
df['Price_in_taka'] = df['Price_in_taka'].astype(float)


# 4. Clean
df = df.dropna()
df = df.drop_duplicates()
df = df.drop('Title', axis=1)


# 5. Outlier remove
q1 = df['Price_in_taka'].quantile(0.25)
q3 = df['Price_in_taka'].quantile(0.75)
iqr = q3 - q1

df = df[(df['Price_in_taka'] > q1 - 1.5*iqr) & (df['Price_in_taka'] < q3 + 1.5*iqr)]


# 6. Log transform
df['Price_in_taka'] = np.log1p(df['Price_in_taka'])


# 7. Split
X = df.drop('Price_in_taka', axis=1)
y = df['Price_in_taka']


# 8. Encoding (NO grouping ❗)
X = pd.get_dummies(X, drop_first=True)


# 9. Train-test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


# 10. BEST MODEL (Balanced XGBoost)
from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(X_train, y_train)


# 11. Predict
y_pred = model.predict(X_test)


# 12. Evaluate
from sklearn.metrics import r2_score

print("R2 Score:", round(r2_score(y_test, y_pred), 2))

R2 Score: 0.85
